# <font color='#1B3A5C'>Modelos de Deep Learning: LSTM y GRU multiserie</font>

> Tercer escalón de la progresión de modelado, tras el baseline estadístico (SARIMA/SARIMAX, notebook 05) y el gradient boosting global (LightGBM y XGBoost, notebooks 06 y 07). Se entrena una red recurrente única sobre las 30 series limpias a la vez (un solo modelo global multiserie), con el mismo split temporal, los mismos horizontes y el mismo arnés de backtesting que el resto del estudio, de modo que la comparación sea directa.

> El modelado es autorregresivo: la red aprende el patrón de cada barrio a partir de su propio histórico de consumo, sin variables exógenas. La decisión se apoya en dos hallazgos previos del propio trabajo, que la autorregresión domina la predicción (análisis SHAP del notebook 07) y que las exógenas climáticas no mejoraron al modelo clásico (SARIMA frente a SARIMAX). El LSTM con exógenas queda como línea futura.


### Qué hace este notebook
1. Backend de deep learning y diagnóstico del entorno (Keras 3, CPU/GPU).
2. Carga del dataset y preparación en formato ancho para la red recurrente.
3. Auditoría de nulos e imputación (las redes no toleran NaN, a diferencia de los árboles).
4. Escalado de las series (obligatorio en redes neuronales).
5. Modelo base LSTM, backtesting en validación y control de sobreajuste.
6. Ajuste de hiperparámetros y variante GRU.
7. Evaluación en test y comparación con el resto de familias.


---
# <font color='#1B3A5C'>0. Backend de deep learning y diagnóstico</font>

> `skforecast 0.22` apoya su `ForecasterRnn` en Keras 3, que necesita un backend de cálculo (TensorFlow, PyTorch o JAX). El contenedor no trae ninguno, así que se instala. Se usa la versión de CPU porque el contenedor no expone la GPU del equipo (el `docker-compose` no la pasa), y el volumen de datos no la requiere.


In [ ]:
# Ejecutar UNA sola vez y luego comentar esta celda (no hace falta reinstalar en cada arranque).
# TensorFlow es el backend mejor probado con skforecast. Versión CPU.
# Si pip intenta DEGRADAR numpy o muestra conflictos de dependencias, parar y avisar:
# en ese caso cambiamos al backend de PyTorch, que no toca numpy.
!pip install "tensorflow-cpu"

> Diagnóstico del entorno. Confirma qué backend y versión hay, si la GPU es visible, y las firmas exactas de las piezas de `ForecasterRnn` en esta versión de skforecast, para construir el modelo sin suposiciones.


In [ ]:
import os, inspect
import numpy as np
import skforecast
print("numpy      :", np.__version__)
print("skforecast :", skforecast.__version__)

# Backend de cálculo. Probamos TensorFlow; si no está, lo reportamos.
try:
    import tensorflow as tf
    print("tensorflow :", tf.__version__)
    gpus = tf.config.list_physical_devices('GPU')
    print("GPU (TF)   :", gpus if gpus else "ninguna, se usa CPU")
except Exception as e:
    print("tensorflow : NO disponible ->", repr(e))

import keras
print("keras      :", keras.__version__, "| backend:", keras.backend.backend())

# Firmas de las piezas clave de ForecasterRnn en skforecast 0.22
from skforecast.deep_learning import ForecasterRnn, create_and_compile_model
print("\ncreate_and_compile_model")
print("  ", inspect.signature(create_and_compile_model))
print("\nForecasterRnn.__init__")
print("  ", inspect.signature(ForecasterRnn.__init__))